Do the prompts differ in whether the AI is exactly correct or not?

In [2]:
import pandas as pd
from statsmodels.stats.contingency_tables import cochrans_q

# Load data
df = pd.read_csv("Output_extraction/ai_grading_final_v3.csv")

# Clean columns
df["answer_key_id"] = df["answer_key_id"].astype(str).str.strip()
df["prompt_type"] = df["prompt_type"].astype(str).str.strip()

# Check prompt names
print(df["prompt_type"].unique())

# Binary outcome: exact correct estimate or not
df["correct_estimate"] = (
    df["ai_estimated_mistakes"] == df["true_mistakes"]
).astype(int)

# Create repetition number because answer_key_id is not always unique
df["rep"] = df.groupby(
    ["answer_key_id", "true_mistakes", "prompt_type"]
).cumcount()

# Convert to wide format
df_wide = df.pivot_table(
    index=["answer_key_id", "true_mistakes", "rep"],
    columns="prompt_type",
    values="correct_estimate",
    aggfunc="first"
)

print(df_wide.columns.tolist())

# Prompt order
prompt_order = [
    "very_pessimistic",
    "pessimistic",
    "neutral",
    "confident",
    "very_confident"
]

# Keep only rows where all five prompt types are present
df_wide = df_wide[prompt_order].dropna()

# Cochran's Q-test
result = cochrans_q(df_wide)

print(result)

['very_pessimistic' 'pessimistic' 'neutral' 'confident' 'very_confident']
['confident', 'neutral', 'pessimistic', 'very_confident', 'very_pessimistic']
df          4
pvalue      0.36483596469043733
statistic   4.31672100605496
